In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv

# Initialize your local environment variables
load_dotenv()

# Setup base foundational model instance
model = ChatOllama(
    model="llama3.2:3b",
    temperature=0.2  # Slightly raised for culinary creativity
)

## 1. Outil de recherche Web (Tavily API)

In [2]:
tavily_client = TavilyClient()

@tool
def recherche_culinaire(query: str) -> Dict[str, Any]:
    """Recherche des recettes de cuisine, des associations d'ingrédients ou des techniques culinaires sur le web.
    
    Args:
        query: la requête de recherche textuelle
    """
    print(f"[Chef Engine] Recherche internet en cours pour : '{query}'...")
    return tavily_client.search(query)

## 2. Configuration du Chef Cuisinier Personnel et Compilation

In [3]:
chef_prompt = """
Vous êtes un Chef Cuisinier Professionnel et un conseiller culinaire personnel de premier ordre.
Votre objectif est d'aider les utilisateurs à concevoir des plats savoureux basés sur les ingrédients disponibles dans leur réfrigérateur.

Directives strictes :
1. Prenez grand soin de respecter les restrictions alimentaires, allergies ou préférences déclarées par l'utilisateur.
2. Utilisez l'outil 'recherche_culinaire' pour valider les étapes d'une recette, trouver des proportions précises ou chercher des inspirations si les ingrédients sont limités.
3. Proposez toujours un ou plusieurs plats clairs et adaptés, suivis d'une brève explication du choix.
4. Répondez de manière concise, polie et structurée en Markdown.
"""

# Compile the final agent graph with full tools and an active checkpointer layer
chef_agent = create_agent(
    model=model,
    tools=[recherche_culinaire],
    system_prompt=chef_prompt,
    checkpointer=InMemorySaver()
)
print("[Chef Engine] Agent Chef Personnel compilé avec succès et prêt à cuisiner.")

[Chef Engine] Agent Chef Personnel compilé avec succès et prêt à cuisiner.


## 3. Session Interactive - Fourniture des ingrédients et préférences

In [4]:
# Set a unique conversation session profile ID
session_config = {"configurable": {"thread_id": "culinary_session_77"}}

# Step 1: Tell the chef what we have and our dietary constraint
user_input_1 = HumanMessage(
    content="Dans mon frigo, j'ai : des tomates, des œufs, du poulet et des épinards. Attention, je mange strictement sans gluten !"
)

print(f"Utilisateur : {user_input_1.content}\n")

response_1 = chef_agent.invoke({"messages": [user_input_1]}, session_config)
print(response_1['messages'][-1].content)

Utilisateur : Dans mon frigo, j'ai : des tomates, des œufs, du poulet et des épinards. Attention, je mange strictement sans gluten !

```json
{"name": "recherche_culinaire", "parameters": {"query": "{\"type\":\"plats de poulet sans gluten\"}"}}
```

Cette requête permettra de trouver des recettes de plats de poulet qui ne contiennent pas de gluten, ce qui est essentiel pour une alimentation sans gluten.


## 4. Validation de la mémoire (Rappel du contexte de session)

In [5]:
# Step 2: Query the agent regarding the previous choices to verify memory persistence
user_input_2 = HumanMessage(
    content="Donne-moi la recette détaillée étape par étape pour la première option que tu as proposée. Rappelle-moi brièvement ma restriction pour confirmer que tu t'en souviens."
)

print(f"Utilisateur : {user_input_2.content}\n")

response_2 = chef_agent.invoke({"messages": [user_input_2]}, session_config)
print(response_2['messages'][-1].content)

Utilisateur : Donne-moi la recette détaillée étape par étape pour la première option que tu as proposée. Rappelle-moi brièvement ma restriction pour confirmer que tu t'en souviens.

Je m'en souviens ! Vous avez une restriction alimentaire pour éviter le gluten. Voici une recette de plat de poulet qui respecte cette restriction :

**Plat de Poulet aux Tomates et Épinards**

Ingrédients :

* 1 poulet de 500g (cuit et détrit)
* 2 tomates fraîches
* 200g d'épinards frais
* 2 cuillères à soupe d'huile d'olive
* Sel et poivre

Étapes :

1. **Préparation du poulet** : Faites cuire le poulet dans une casserole avec un peu d'eau et de sel, jusqu'à ce qu'il soit tendre. Laissez-le refroidir.
2. **Préparation des tomates et des épinards** : Hachez les tomates en petits morceaux et lavez les épinards frais. Égouttez-les et séchez-les avec une serviette.
3. **Préparation de la sauce** : Dans une poêle, chauffez l'huile d'olive à feu moyen. Ajoutez les tomates hachées et laissez-les cuire pendant 5 